# آماده‌سازی دادهٔ پلاک فارسی
این نوت‌بوک را برای هر نسخهٔ دیتاست پیش از نوت‌بوک اصلی با Run All اجرا کن.
سلامت JPG/XML و کادرها و تکرار/شباهت را بررسی می‌کند؛ مدلی آموزش نمی‌بیند.
از هر گروه یک تصویر در دیتاست باقی می‌ماند. یک کپی از آن با پسوند original و
بقیه با پسوند duplicateN_split، همراه XML، در duplicate_photos بایگانی می‌شوند.
پس از تأیید هش تمام کپی‌ها و ثبت manifest.csv، فقط جفت‌های اضافی از دیتاست حذف می‌شوند.
original یعنی عضو منتخب گروه، نه لزوماً عکس اولیهٔ دوربین یا باکیفیت‌ترین عکس.
گروه‌ها شامل تکرار دقیق و شباهت تقریبی‌اند؛ پوشهٔ بایگانی برای بازبینی حفظ می‌شود.
کادرها یا تصاویر نامعتبر فقط گزارش می‌شوند و در این مرحله جابه‌جا نمی‌شوند.
خروجی جدول‌های آماده در prepared_data است. با تغییر دیتاست، آماده‌سازی را تکرار کن.
مقایسهٔ جفت‌ها زمان‌بر است و استقلال خودرو/صحنه را تضمین نمی‌کند.


In [ ]:
import os
import hashlib
import shutil
from datetime import datetime
import xml.etree.ElementTree as ET

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display

from predict import PLATE_LABEL, file_hash

# Run from the project root; keep the review archive outside the scanned dataset.
ROOT = os.getcwd()
DATA_FOLDER = os.path.join(ROOT, "new_dataset")
PREPARED_FOLDER = os.path.join(ROOT, "prepared_data")
DUPLICATE_FOLDER = os.path.join(ROOT, "duplicate_photos")
# Keep class names and order consistent with the training notebook and XML labels.
CLASSES = list("0123456789") + ["الف", "ب", "ت", "ث", "ج", "د", "س", "ش", "ص", "ط", "ظ",
                                "ع", "ق", "ل", "م", "ن", "ه\u200d", "و", "پ", "ژ (معلولین و جانبازان)", "ی"]
if not os.path.isdir(DATA_FOLDER):
    raise FileNotFoundError("Open this notebook from the project folder containing new_dataset.")


## ۱. بررسی تصاویر، XMLها و کادرها
ابعاد واقعی تصویر مبنا هستند؛ خطاها و هشدارها جدا ثبت می‌شوند.


In [ ]:
# Audit source files and return valid image/box tables plus similarity descriptors.
def scan_dataset():
    images, boxes, issues, thumbnails, hashes = [], [], [], [], []
    for split in ("train", "validation", "test"):
        folder = os.path.join(DATA_FOLDER, split)
        if not os.path.isdir(folder):
            raise FileNotFoundError(folder)
        paths = []
        for current_folder, _, names in os.walk(folder):
            paths.extend(os.path.join(current_folder, name) for name in names)
        # Pair files by their full path without extension; never rely on directory listing order.
        image_paths = sorted(p for p in paths if p.lower().endswith((".jpg", ".jpeg")))
        xml_paths = {os.path.splitext(p)[0]: p for p in paths if p.lower().endswith(".xml")}
        paired = set()
        for path in image_paths:
            xml_path = xml_paths.get(os.path.splitext(path)[0])
            if xml_path is None:
                issues.append({"path": path, "reason": "missing_xml", "severity": "error"})
                continue
            # Track pairing separately from validity so corrupt pairs are not reported as missing images.
            paired.add(xml_path)
            image = cv2.imread(path)
            try:
                if image is None:
                    raise ValueError("unreadable_image")
                # Use Pillow verification in addition to OpenCV decoding to catch file-structure errors.
                with Image.open(path) as opened:
                    opened.verify()
                # Use decoded image dimensions rather than potentially stale XML metadata.
                height, width = image.shape[:2]
                annotation = ET.parse(xml_path).getroot()
                image_boxes = []
                for obj in annotation.findall("object"):
                    label = (obj.findtext("name") or "").strip()
                    coords = [float(obj.findtext("bndbox/" + name, "nan"))
                              for name in ("xmin", "ymin", "xmax", "ymax")]
                    x1, y1, x2, y2 = coords
                    # Require finite, positive-area boxes fully inside the image and a known class label.
                    if not label or not np.isfinite(coords).all() or not (0 <= x1 < x2 <= width and 0 <= y1 < y2 <= height):
                        raise ValueError("invalid_label_or_box")
                    if label != PLATE_LABEL and label not in CLASSES:
                        raise ValueError("unknown_character_class: " + label)
                    image_boxes.append({"label": label, "x1": x1, "y1": y1, "x2": x2, "y2": y2})
                if not any(b["label"] == PLATE_LABEL for b in image_boxes):
                    raise ValueError("missing_plate")
            # Quarantine the entire pair on invalid data instead of training with partially missing labels.
            except (OSError, ValueError, ET.ParseError) as error:
                issues.append({"path": path, "reason": str(error), "severity": "error"})
                continue
            # Metadata mismatches are warnings after actual image dimensions and boxes have passed validation.
            size = annotation.find("size")
            if size is not None and (size.findtext("width") != str(width) or size.findtext("height") != str(height)):
                issues.append({"path": xml_path, "reason": "stale_xml_dimensions", "severity": "warning"})
            if annotation.findtext("filename") != os.path.basename(path):
                issues.append({"path": xml_path, "reason": "stale_xml_filename", "severity": "warning"})
            image_id = len(images)
            # File hashes detect later edits; decoded-pixel hashes detect exact visual duplicates.
            images.append({"id": image_id, "split": split,
                           "image": os.path.relpath(path, ROOT).replace(os.sep, "/"),
                           "xml": os.path.relpath(xml_path, ROOT).replace(os.sep, "/"),
                           "width": width, "height": height, "image_sha256": file_hash(path),
                           "xml_sha256": file_hash(xml_path),
                           "pixel_sha256": hashlib.sha256(image.tobytes() + str(image.shape).encode()).hexdigest()})
            boxes.extend({"image_id": image_id, **box} for box in image_boxes)
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
            # A 9x8 grayscale thumbnail yields 64 horizontal comparisons for a compact dHash.
            tiny = cv2.resize(gray, (9, 8), interpolation=cv2.INTER_AREA)
            hashes.append(sum(int(bit) << i for i, bit in enumerate((tiny[:, 1:] > tiny[:, :-1]).flat)))
            # Retain a second thumbnail for brightness-difference checks after the hash prefilter.
            thumbnails.append(cv2.resize(gray, (32, 32), interpolation=cv2.INTER_AREA))
        # Report XML files that have no corresponding JPG/JPEG, independently of image-side errors.
        for xml_path in set(xml_paths.values()) - paired:
            issues.append({"path": xml_path, "reason": "missing_image", "severity": "error"})
        print(split, "scanned")
    if not images:
        raise ValueError("No valid image/XML pairs found.")
    return pd.DataFrame(images), pd.DataFrame(boxes), pd.DataFrame(issues), hashes, thumbnails

## ۲. گروه‌بندی تکرار و شباهت
از هر گروه یک تصویر با اولویت test، سپس validation و بعد train نگه داشته می‌شود.
آستانه‌ها: اختلاف حداکثر ۴ بیت dHash، اختلاف نسبت ابعاد حداکثر ۳٪ و اختلاف متوسط روشنایی حداکثر ۸.


In [ ]:
def filter_similar_images(images, hashes, thumbnails):
    # Union-find connects matching images into groups; initially each image is its own root.
    parents = list(range(len(images)))
    def group_of(i):
        while parents[i] != i:
            # Compress parent paths so later group lookups are faster.
            parents[i] = parents[parents[i]]
            i = parents[i]
        return i

    exact_images, unique_ids = {}, []
    records = images.to_dict("records")
    for i, record in enumerate(records):
        pixel_hash = record["pixel_sha256"]
        # Exact pixel matches can join an existing group without repeating approximate comparisons.
        if pixel_hash in exact_images:
            parents[group_of(i)] = group_of(exact_images[pixel_hash])
            continue
        for j in unique_ids:
            # Require at most four differing dHash bits before the more expensive checks.
            if (hashes[i] ^ hashes[j]).bit_count() > 4:
                continue
            ratio = record["width"] / record["height"]
            other_ratio = records[j]["width"] / records[j]["height"]
            # Reject aspect-ratio differences greater than 3%.
            if abs(ratio / other_ratio - 1) > 0.03:
                continue
            # Use float pixels to avoid uint8 subtraction overflow; mean difference must be at most 8.
            difference = np.abs(thumbnails[i].astype(np.float32) - thumbnails[j]).mean()
            # Link similar pairs; transitive grouping may join images that are not directly similar.
            if difference <= 8:
                parents[group_of(i)] = group_of(j)
        exact_images[pixel_hash] = i
        unique_ids.append(i)
        if (i + 1) % 2000 == 0:
            print("Similarity checked:", i + 1, "/", len(records))

    # Keep one member per group, preferring test, then validation, then train; break ties by path.
    priority = {"test": 0, "validation": 1, "train": 2}
    best = {}
    for i, record in enumerate(records):
        group = group_of(i)
        key = (priority[record["split"]], record["image"])
        if group not in best or key < best[group][0]:
            best[group] = (key, i)
    keep = {value[1] for value in best.values()}
    images["keep"] = [int(i in keep) for i in range(len(images))]
    # Use the retained member position as a group identifier for review and archiving.
    images["duplicate_group"] = [best[group_of(i)][1] for i in range(len(images))]
    return images


## ۳. بایگانی تصاویر اضافی
برای هر گروه، original کپی می‌شود و duplicateN_train/validation/test همراه XML منتقل می‌شوند.
manifest.csv مسیر قبلی و جدید، هش و وضعیت هر فایل را ثبت می‌کند. در خطای میان اجرا، قبل از تکرار آن را بررسی کن؛ کپی‌های تأییدشده باقی می‌مانند.


In [ ]:
# Copy one retained original per group and move surplus JPG/XML pairs into a review archive.
def archive_duplicates(images):
    duplicates = images[images["keep"] == 0]
    if duplicates.empty:
        print("No duplicates to move.")
        return None

    # Resolve paths before any deletion and reject an archive nested inside the scanned dataset.
    data_root = os.path.realpath(DATA_FOLDER)
    archive_root = os.path.realpath(DUPLICATE_FOLDER)
    if os.path.commonpath([data_root, archive_root]) == data_root:
        raise ValueError("duplicate_photos must be outside new_dataset.")
    # Use a fresh timestamped directory to preserve previous archives and avoid overwrites.
    run_folder = os.path.join(archive_root, datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
    os.makedirs(run_folder, exist_ok=False)
    records = []
    seen = set()
    for group_id in sorted(duplicates["duplicate_group"].unique()):
        group = images[images["duplicate_group"] == group_id]
        original = group[group["keep"] == 1]
        if len(original) != 1:
            raise ValueError("Expected one retained image per group.")
        # Put the retained member first so duplicate numbering is stable within this run.
        members = pd.concat([original, group[group["keep"] == 0].sort_values("image")])
        group_folder = os.path.join(run_folder, f"group_{int(group_id):06d}")
        os.makedirs(group_folder)
        for number, row in enumerate(members.to_dict("records")):
            role = "original" if number == 0 else f"duplicate{number}"
            stem = os.path.splitext(os.path.basename(row["image"]))[0]
            for kind in ("image", "xml"):
                source = os.path.realpath(os.path.join(ROOT, row[kind]))
                # Only plan operations on distinct resolved source paths inside the intended dataset.
                if os.path.commonpath([data_root, source]) != data_root or source in seen:
                    raise ValueError("Unsafe or repeated source path: " + source)
                seen.add(source)
                expected_hash = row[kind + "_sha256"]
                # Abort if a source changed between scanning and archive planning.
                if file_hash(source) != expected_hash:
                    raise ValueError("Source changed since scan: " + source)
                extension = os.path.splitext(source)[1]
                destination = os.path.join(group_folder, f"{stem}_{role}_{row['split']}{extension}")
                records.append({"group": int(group_id), "role": role, "split": row["split"],
                                "kind": kind, "source": source, "destination": destination,
                                "sha256": expected_hash, "status": "pending"})

    # Record source/destination paths and hashes so interrupted transfers can be inspected and restored.
    manifest = os.path.join(run_folder, "manifest.csv")
    def save_manifest():
        pd.DataFrame(records).to_csv(manifest, index=False, encoding="utf-8-sig")

    save_manifest()
    # Finish and verify every backup before deleting any source file.
    for record in records:
        if os.path.exists(record["destination"]):
            raise FileExistsError(record["destination"])
        shutil.copy2(record["source"], record["destination"])
        if file_hash(record["destination"]) != record["sha256"]:
            raise ValueError("Backup checksum mismatch: " + record["destination"])
        record["status"] = "copied"
    save_manifest()
    for record in records:
        if record["role"] == "original":
            continue
        # Recheck both copies immediately before each individual deletion.
        if (file_hash(record["source"]) != record["sha256"]
                or file_hash(record["destination"]) != record["sha256"]):
            raise ValueError("File changed before removal: " + record["source"])
        # Remove only verified surplus files; the retained original is never deleted.
        os.remove(record["source"])
        record["status"] = "moved"
        # Persist progress after each removal; a pair can be partially moved if execution is interrupted.
        save_manifest()
    print("Duplicate images moved:", len(duplicates), "| Review:", run_folder)
    return manifest


## ۴. اجرا و ذخیرهٔ دادهٔ آماده
جدول‌ها پیش از انتقال ذخیره می‌شوند؛ keep=0 سابقهٔ تصاویر کنارگذاشته‌شده را نگه می‌دارد.
پس از انتقال، مسیر قبلی آن‌ها در دیتاست وجود ندارد؛ محل کپی در manifest ثبت شده است.


In [ ]:
# Audit first, then group duplicates; no model training or metric-based tuning happens here.
images, boxes, issues, hashes, thumbnails = scan_dataset()
images = filter_similar_images(images, hashes, thumbnails)
os.makedirs(PREPARED_FOLDER, exist_ok=True)
# Save the full index before archiving so excluded rows remain traceable with keep=0.
images.to_csv(os.path.join(PREPARED_FOLDER, "images.csv"), index=False, encoding="utf-8-sig")
boxes.to_csv(os.path.join(PREPARED_FOLDER, "boxes.csv"), index=False, encoding="utf-8-sig")
issues.to_csv(os.path.join(PREPARED_FOLDER, "issues.csv"), index=False, encoding="utf-8-sig")
# This step modifies the dataset: surplus pairs are removed only after verified backups exist.
manifest_path = archive_duplicates(images)
# The main notebook consumes retained rows only; excluded source paths may no longer exist.
selected = images[images["keep"] == 1]
display(selected.groupby("split").size().rename("images").to_frame())
print("Valid images:", len(images), "| Kept:", len(selected), "| Excluded:", len(images) - len(selected))
if not issues.empty:
    display(issues.groupby(["severity", "reason"]).size().rename("count").to_frame())
print("Ready: open Iranian_ALPR.ipynb. Duplicate pairs were archived; retained originals remain in the dataset.")
